# Этап 6 — Основная модель (CatBoost)

**Цель:** обучить градиентный бустинг (CatBoost) и проверить, побьёт ли он бейзлайн из этапа 5 (LogisticRegression). Сравнение — на **той же** тестовой выборке и по тем же метрикам.

**Бейзлайн для сравнения (этап 5, датасет 3072):** ROC-AUC 0.692 · Precision 0.576 · Recall 0.642 · F1 0.607.

**Логика ноутбука:** CatBoost «из коробки» → диагностика переобучения → лечение (early stopping) → тюнинг (grid) → сравнение с бейзлайном и вывод.

In [66]:
import sys
sys.path.append('../src')   # чтобы видеть модули из src/

import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from evaluation import evaluate   # функция оценки метрик вынесена в src/evaluation.py

## 1. Данные и подготовка признаков

Загружаем `churn_dataset.csv` из этапа 4. Особенности подготовки под CatBoost:

- **не нужен one-hot и масштабирование** — деревья работают с признаками как есть (в отличие от логрега на этапе 5);
- `Cluster` (число) выбрасываем, оставляем текстовый `Cluster_name` и объявим его категорией через `cat_features`;
- `Customer ID` — в индекс: не признак, но связь с клиентом сохранена.

Разбиение train/test — **с теми же параметрами, что и бейзлайн** (`test_size=0.2, random_state=101, stratify=y`), иначе сравнение будет нечестным (модели должны проверяться на одних и тех же клиентах).

In [67]:
df = pd.read_csv('../data/processed/churn_dataset.csv')
df = df.set_index('Customer ID')
df.head()

,Recency,Frequency,Monetary,churn,Cluster,Cluster_name
Customer ID,,,,,,
12349.0,115,3,1244.37,0,2,Обычные
12355.0,112,1,488.21,1,0,Новые
12358.0,95,2,1697.93,0,2,Обычные
12359.0,80,7,1918.03,0,1,Лояльные
12360.0,108,4,740.04,0,2,Обычные


In [68]:
y = df['churn']
# Cluster (число) убираем, оставляем Cluster_name как категорию; churn — целевая переменная
X = df.drop(columns=['Cluster', 'churn'])

In [69]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=101, stratify=y
)

## 2. Оценка метрик

Функция `evaluate(model, X, y)` вынесена в `src/evaluation.py` (переиспользуется на этапах 5–7): считает 4 метрики и возвращает словарь (ROC-AUC на вероятностях, остальные на метках). Накопление результатов в `results` — логика этого ноутбука, для итоговой сравнительной таблицы.

In [70]:
# словарь для накопления метрик всех моделей (для сравнительной таблицы ниже)
results = {}

## 3. CatBoost «из коробки»

Первый заход — дефолтные гиперпараметры (1000 деревьев), без тюнинга. Цель — получить стартовую точку и сравнить с бейзлайном.

In [71]:
model_default = CatBoostClassifier(
    cat_features=['Cluster_name'],
    random_state=101,
    verbose=200,
)
model_default.fit(X_train, y_train)

Learning rate set to 0.015123
0:	learn: 0.6904024	total: 44.7ms	remaining: 44.6s
200:	learn: 0.5949800	total: 9.08s	remaining: 36.1s
400:	learn: 0.5841661	total: 17.8s	remaining: 26.6s
600:	learn: 0.5712049	total: 27s	remaining: 17.9s
800:	learn: 0.5511599	total: 36.5s	remaining: 9.06s
999:	learn: 0.5334934	total: 46s	remaining: 0us


CatBoostClassifier(cat_features=['Cluster_name'], random_state=101, verbose=200)

In [72]:
results['CatBoost (default)'] = evaluate(model_default, X_test, y_test)
results['CatBoost (default)']

{'ROC-AUC': 0.689, 'Precision': 0.587, 'Recall': 0.613, 'F1': 0.599}

**Наблюдение:** метрики на test оказались **примерно на уровне бейзлайна** (ROC-AUC ~0.686 против 0.692), не лучше. Мощная модель не дала выигрыша.

Заодно проверим, нет ли **переобучения**: 1000 деревьев на 4 признаках и ~3000 строках — это много, модель может «зубрить» train. Сравним метрики на train и на test.

## 4. Диагностика переобучения — train vs test

Если модель переобучена, метрики на train будут заметно выше, чем на test.

In [73]:
# train-метрики не сохраняем в results (это диагностика, не строка сравнения)
train_m = evaluate(model_default, X_train, y_train)
test_m = results['CatBoost (default)']

print('TRAIN:', train_m)
print('TEST: ', test_m)

TRAIN: {'ROC-AUC': 0.801, 'Precision': 0.686, 'Recall': 0.663, 'F1': 0.674}
TEST:  {'ROC-AUC': 0.689, 'Precision': 0.587, 'Recall': 0.613, 'F1': 0.599}


**Вывод:** на train ROC-AUC ≈ 0.84, на test ≈ 0.69 — разрыв ~0.15. **Переобучение подтверждено:** модель хорошо помнит обучающие данные, но плохо обобщает на новые.

Лечим — ограничим число деревьев через **early stopping**.

## 5. Лечение — early stopping

Идея: остановить обучение, когда качество на отложенных данных перестаёт расти, вместо того чтобы жечь все 1000 деревьев.

**Важно:** для контроля нужна **валидационная** выборка, а **не** test. Если останавливаться по test, мы используем его для настройки модели → утечка, и test перестаёт быть честной финальной проверкой. Поэтому режем ещё кусок **от train**:

- `X_tr` — учим деревья;
- `X_val` — контроль (early stopping);
- `X_test` — только финальная оценка, не трогаем до конца.

`stratify=y_train` — сохраняем баланс классов (режем именно train, поэтому от `y_train`, не от полного `y`).

In [74]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=101, stratify=y_train
)

In [75]:
model_es = CatBoostClassifier(
    cat_features=['Cluster_name'],
    random_state=101,
    early_stopping_rounds=50,   # если 50 итераций подряд val не улучшается — стоп
    use_best_model=True,        # взять лучшую итерацию, а не последнюю
    verbose=100,
)
model_es.fit(X_tr, y_tr, eval_set=(X_val, y_val))

Learning rate set to 0.037443
0:	learn: 0.6867177	test: 0.6865322	best: 0.6865322 (0)	total: 39.7ms	remaining: 39.7s
100:	learn: 0.5911476	test: 0.6009373	best: 0.5995819 (76)	total: 4.49s	remaining: 39.9s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5995819476
bestIteration = 76

Shrink model to first 77 iterations.


CatBoostClassifier(cat_features=['Cluster_name'], early_stopping_rounds=50, random_state=101, use_best_model=True, verbose=100)

In [76]:
results['CatBoost (early stop)'] = evaluate(model_es, X_test, y_test)
results['CatBoost (early stop)']

{'ROC-AUC': 0.695, 'Precision': 0.595, 'Recall': 0.59, 'F1': 0.593}

## 6. Тюнинг гиперпараметров — GridSearchCV

Ручной подбор заменяем на перебор по сетке с кросс-валидацией. Дисциплина: **CV только на train**, финальная оценка лучшей модели — на нетронутом test.

Ожидание: мы **data-bound** (потолок задают 4 признака), поэтому grid, скорее всего, не побьёт логрег — но это честная проверка «а вдруг тюнинг спасёт бустинг», и готовый каркас на будущее (когда признаков станет больше, здесь же перезапустим сетку).

In [78]:
model_grid = CatBoostClassifier(
    cat_features=['Cluster_name'],
    iterations=300,
    random_state=101,
    verbose=0,
)

param_grid = {
    'depth': [3, 4, 6],
    'l2_leaf_reg': [3, 10],
    'learning_rate': [0.03, 0.1],
}

grid_result = model_grid.grid_search(param_grid, X=X_train, y=y_train, cv=3, verbose=0)
print('Лучшие параметры:', grid_result['params'])



bestTest = 0.6188160224
bestIteration = 85


bestTest = 0.616120726
bestIteration = 34


bestTest = 0.6194898756
bestIteration = 89


bestTest = 0.6191320334
bestIteration = 25


bestTest = 0.6181518208
bestIteration = 76


bestTest = 0.6188740272
bestIteration = 27


bestTest = 0.6176335988
bestIteration = 83


bestTest = 0.6181634249
bestIteration = 21


bestTest = 0.6189809491
bestIteration = 82


bestTest = 0.6205129067
bestIteration = 25


bestTest = 0.6195846573
bestIteration = 68


bestTest = 0.6199638239
bestIteration = 24

Training on fold [0/3]

bestTest = 0.61148518
bestIteration = 34

Training on fold [1/3]

bestTest = 0.600043891
bestIteration = 59

Training on fold [2/3]

bestTest = 0.6216634811
bestIteration = 22

Лучшие параметры: {'depth': 3, 'learning_rate': 0.1, 'l2_leaf_reg': 3}


In [80]:
results['CatBoost (grid)'] = evaluate(model_grid, X_test, y_test)
results['CatBoost (grid)']

{'ROC-AUC': 0.686, 'Precision': 0.577, 'Recall': 0.605, 'F1': 0.591}

## 7. Итог — сравнение с бейзлайном

Все метрики на **одной и той же** тестовой выборке.

In [ ]:
baseline = {'ROC-AUC': 0.692, 'Precision': 0.576, 'Recall': 0.642, 'F1': 0.607}
comparison = pd.DataFrame({'LogReg (baseline)': baseline, **results}).T
comparison

## Вывод

Early stopping убрал переобучение (модель остановилась ~на 95 деревьях вместо 1000), grid-тюнинг подобрал гиперпараметры — но **ни один вариант CatBoost не оторвался от бейзлайна**.

Ключевая метрика — **ROC-AUC** (не зависит от порога, меряет качество ранжирования по риску оттока). Все модели сгрудились в узком коридоре **~0.686–0.696**:

| Модель | ROC-AUC |
|---|---|
| LogReg | 0.692 |
| CatBoost default | 0.686 |
| CatBoost early stop | 0.696 |
| CatBoost grid | 0.686 |

Разброс — **в пределах шума**: ни одна модель не выигрывает значимо. Это честный и важный результат: для задачи с **4 признаками** и **~3000 строк**, где RFM-зависимости довольно линейны, **сложная модель не бьёт простую**. Мы **data-bound** — потолок задают признаки, а не сложность модели (все упёрлись в ~0.69).

**Выбор финальной модели — LogisticRegression**, по **бритве Оккама**: при равном качестве берём более простую и интерпретируемую модель (важно для SHAP на этапе 7 и объяснимости в демо). CatBoost честно проверен — из коробки, с early stopping и с grid — и отклонён не «на глаз», а по метрикам.

**Что дальше:**

1. Подбор порога классификации под бизнес-цель (**Recall**) — сделан в `05_baseline_model.ipynb` на логреге.
2. (в README) feature engineering — тренд активности, tenure, сезонность — поднимет потолок сильнее, чем смена модели. Тогда же имеет смысл перезапустить grid: на богатых признаках бустинг может обойти логрег.